# M6: Cosmos Predict — Synthetic Scenario Generation

**Stage 5 (extension): Synthetic Scenario Generation — Cosmos Predict 2.5** *(supplementary to the 8-stage blog pipeline)*

| Item | Detail |
|------|--------|
| Model | Cosmos-Predict2.5-2B (base, Video2World) |
| Instance | Adapts to your GPU: **g5/g6.12xlarge+** run 480x832 (multi-GPU shard); **p4d.24xlarge** (A100) / **p5.48xlarge** (H100) run native resolution. The notebook auto-detects and picks the strategy. |
| Input | nuScenes CAM_FRONT clip built in **M5** (`m5/source/`), or reassembled from **M1** |
| Output | `/users/{profile}/m6/` (synthetic future-frame scenarios) |
| Repo | [nvidia-cosmos/cosmos-predict2.5](https://github.com/nvidia-cosmos/cosmos-predict2.5) |

This module uses the Cosmos Predict 2.5 **world model** in **Video2World** mode:
given a real driving clip as context, it predicts plausible future frames under
a scenario prompt (near-collision, pedestrian crossing, sudden brake, lane
cut-in) — synthetic edge cases for AV safety validation.

### How this notebook actually runs Cosmos Predict

Like M5, there is **no `pip install cosmos-predict2`** — the real workflow clones
the official repo, `uv sync`s its pinned deps, and calls `examples/inference.py`.
Predict 2.5 uses a **separate** package/venv from Transfer 2.5 (`cosmos_predict2`
vs `cosmos_transfer2`), so `scripts/setup_cosmos_env.sh` builds a second venv and
writes **`cosmos_predict_env.sh`** (M5 keeps using `cosmos_env.sh`). Base 2B
Video2World needs no post-training checkpoint — just
`examples/inference.py -i <spec.json> -o <out> --inference-type=video2world`.

> **Gated models:** the Cosmos checkpoints need a HuggingFace token whose account
> accepted the licenses on
> [Cosmos-Predict2.5-2B](https://huggingface.co/nvidia/Cosmos-Predict2.5-2B),
> [Cosmos-Guardrail1](https://huggingface.co/nvidia/Cosmos-Guardrail1), and
> [Cosmos-Reason1-7B](https://huggingface.co/nvidia/Cosmos-Reason1-7B). Set
> `HF_TOKEN` in the setup cell.

> **Ephemeral install:** the env lives on the instance NVMe and is reset when the
> JupyterLab app restarts. Re-run the setup cell after any restart — it's
> idempotent (skips the Transfer venv if M5 already built it; only syncs Predict).

In [ ]:
# ============================================================
# Setup — configuration
# ============================================================
import os
import sys
import time
import json
import glob
import subprocess
from pathlib import Path
from datetime import datetime, timezone

import boto3

# ------------------------------------------------------------
# HuggingFace token — needed ONLY if this region has no pre-seeded cache.
# When the admin has seeded s3://<shared>/hf-cache/ in THIS region, the setup cell
# restores it and runs Hugging Face offline, so you leave this blank. When it has
# NOT been seeded, the setup cell stops immediately and says so, rather than failing
# 20 minutes later inside torchrun. In that case either ask the admin to seed the
# region, or accept the licenses on huggingface.co/nvidia/Cosmos-Predict2.5-2B and
# /Cosmos-Guardrail1 (instant, auto-approved) and paste a token here.
# An env var (admin-injected) always takes priority.
# ------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN", "") or ""   # optional; leave "" to use the offline cache
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
AWS_REGION = boto3.session.Session().region_name  # buckets are per-region (see
# infra/av30_constructs/storage.py): the fallback name MUST carry the region, or a
# notebook whose env vars are missing silently derives a bucket that does not exist.
PROFILE = os.environ.get("USER_PROFILE", os.environ.get("BLUEPRINT_PROFILE", "default"))
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}-{AWS_REGION}")
S3_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}-{AWS_REGION}")

# M5 already assembled a nuScenes CAM_FRONT clip and uploaded it here; we reuse
# it. If M5 wasn't run, cell 4 rebuilds it from M1's manifest.
M1_PREFIX = f"users/{PROFILE}/m1/"
M5_SOURCE_KEY = f"users/{PROFILE}/m5/source/nuscenes_cam_front.mp4"
OUTPUT_PREFIX = f"users/{PROFILE}/m6/"
NUSCENES_PREFIX = "datasets/nuscenes-mini/"   # in SHARED_BUCKET

# Local scratch on the instance NVMe (28 TB on p4d/p5).
NVME = "/mnt/sagemaker-nvme" if os.path.isdir("/mnt/sagemaker-nvme") else "/tmp"
WORK = f"{NVME}/m5_work"
FRAMES_DIR = f"{WORK}/frames"
INPUT_MP4 = f"{WORK}/nuscenes_cam_front.mp4"
OUTPUT_DIR = f"{WORK}/out"
SPECS_DIR = f"{WORK}/specs"

# Cosmos Predict env produced by scripts/setup_cosmos_env.sh (SEPARATE from M5's
# cosmos_env.sh — different package/venv).
COSMOS_WORK = f"{NVME}/cosmos-work"
COSMOS_PREDICT_REPO = f"{COSMOS_WORK}/cosmos-predict2.5"
COSMOS_PREDICT_ENV = f"{COSMOS_WORK}/cosmos_predict_env.sh"

# Video assembly params (mirror M5: 57 frames @ 1280x704 validated).
VIDEO_W, VIDEO_H = 1280, 704
VIDEO_FPS = 10
MAX_FRAMES = 57

# Scenario prompts for Video2World future-frame prediction. Each becomes one run.
SCENARIO_PROMPTS = {
    "near_collision": (
        "A realistic dashcam driving scene. The vehicle ahead suddenly brakes "
        "hard and the gap closes rapidly; the ego vehicle must react. Physically "
        "plausible motion, consistent road and lighting, no cartoonish artifacts."
    ),
    "pedestrian_crossing": (
        "A realistic dashcam driving scene. A pedestrian steps into the road from "
        "between parked cars ahead; the ego vehicle slows to yield. Physically "
        "plausible motion, consistent road and lighting."
    ),
    "sudden_brake": (
        "A realistic dashcam driving scene. Traffic ahead comes to an abrupt stop; "
        "brake lights illuminate in sequence. Physically plausible deceleration, "
        "consistent road and lighting."
    ),
    "lane_cut_in": (
        "A realistic dashcam driving scene. An adjacent vehicle cuts into the ego "
        "lane with a small gap; the ego vehicle adjusts speed. Physically plausible "
        "motion, consistent road and lighting."
    ),
}
# Keep a workshop run cheap/fast by default: one scenario. Extend to the full
# list to generate all four.
SCENARIOS = ["near_collision"]

os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPECS_DIR, exist_ok=True)

print(f"Profile: {PROFILE}")
print(f"Input  : {M5_SOURCE_KEY} (M5 clip; else rebuilt from M1)")
print(f"Output : s3://{S3_BUCKET}/{OUTPUT_PREFIX}")
print(f"Work   : {WORK}")
print(f"Scenarios: {SCENARIOS}")
# Report the token state as a fact, not a conclusion: whether the offline cache
# actually exists in THIS region is not known until the setup cell probes S3.
print(f"HF_TOKEN: {'set' if HF_TOKEN else 'not set (setup will verify the admin offline cache)'}")

In [ ]:
# ============================================================
# Pre-flight — detect GPUs and choose an instance-appropriate run strategy
# ============================================================
# Cosmos-Predict2.5-2B (Video2World) needs ~65 GB to load on ONE GPU — the same
# constraint as M5's Transfer model. So we detect PER-GPU VRAM (not the sum) and
# pick a strategy that fits the instance you launched:
#
#   per-GPU >= 70 GB (H100 / p5)    -> single GPU,   native res, guardrails ON
#   per-GPU >= 38 GB (A100 / p4d)   -> N-GPU shard,  native res, guardrails ON
#   per-GPU  < 38 GB (L4/A10G,g5/g6)-> N-GPU shard,  480x832,    guardrails OFF
#
# The N-GPU path uses torchrun context parallelism (shards diffusion activations
# across ranks). On 24 GB cards we also drop resolution + output frames and turn
# guardrails off — completes on a g6 (L4 24GB) when the GPUs are otherwise idle.
#
# IMPORTANT: we probe the GPUs with `nvidia-smi` in a SUBPROCESS, never with
# torch.cuda in this kernel. Touching torch.cuda here creates a CUDA context
# INSIDE the JupyterLab kernel that lives for the whole session, permanently
# pinning GPU memory the torchrun workers then need — a real cause of the VAE
# tokenizer OOM. nvidia-smi reads the driver without a CUDA context, so the
# kernel stays off the GPUs entirely.
import subprocess

FOREIGN_MIB = 1024   # >1 GiB used, with no run in flight, == a foreign process

def _probe_gpus_nvidia_smi():
    """Per-GPU (total_mib, used_mib) via nvidia-smi — creates NO CUDA context."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.total,memory.used",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    ).stdout
    gpus = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        total_s, used_s = (p.strip() for p in line.split(","))
        gpus.append((int(total_s), int(used_s)))   # MiB, MiB
    return gpus


def _probe_compute_apps():
    """Which PROCESSES hold GPU memory — also via nvidia-smi, so still no CUDA context.

    The occupancy halt below used to say "almost always ANOTHER notebook kernel" and leave
    the participant to guess which one. nvidia-smi can name them, so it does.
    """
    out = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,used_memory,process_name",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    apps = []
    for line in (out.stdout or "").strip().splitlines():
        parts = [p.strip() for p in line.split(",")]
        if len(parts) >= 3:
            apps.append((parts[0], parts[1], ",".join(parts[2:])))
    return apps

try:
    _gpus = _probe_gpus_nvidia_smi()
except FileNotFoundError:
    # nvidia-smi absent almost always means the CPU image was launched. Fall back
    # to torch ONLY to produce total VRAM so tiering still works; this DOES create
    # a kernel CUDA context (the thing we avoid above), but a card with no
    # nvidia-smi is not the multi-GPU shard path where that context matters.
    try:
        import torch
        assert torch.cuda.is_available()
        _n = torch.cuda.device_count()
        _gpus = [(int(torch.cuda.get_device_properties(i).total_memory / (1024**2)), 0)
                 for i in range(_n)]
    except Exception as e:
        raise RuntimeError(
            "nvidia-smi not found and no CUDA GPU visible — this looks like the "
            "CPU image. Use Instance Options to pick a GPU instance, then re-run."
        ) from e
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "nvidia-smi failed — this notebook requires a GPU instance. "
        "Use Instance Options to pick a GPU type."
    ) from e

assert _gpus, "nvidia-smi reported no GPUs — this notebook requires a GPU instance."

GPU_COUNT = len(_gpus)                               # rows == device count
per_gpu_gb = [tot / 1024 for tot, _ in _gpus]        # MiB -> GiB (same scale as torch)
per_gpu_used_mib = [used for _, used in _gpus]
MIN_PER_GPU_GB = min(per_gpu_gb)
print(f"GPUs available: {GPU_COUNT}")
for i, (tot, used) in enumerate(_gpus):
    print(f"  GPU {i}: {tot / 1024:.1f} GB total — {used} MiB in use")
print(f"Smallest GPU: {MIN_PER_GPU_GB:.1f} GB")

# --- Halt if ANOTHER process already holds the GPUs ------------------------
# The kernel never touches CUDA (we probe via nvidia-smi), so any significant
# 'in use' memory here belongs to a FOREIGN process — almost always another
# notebook kernel left running, or a stale torchrun rank from an interrupted
# run. Cosmos Predict needs (nearly) the whole card, so leaving it there would
# OOM deep in the VAE tokenizer with an opaque CUDA error AFTER the ~15-20 min
# setup + model load. Detect it and stop cleanly now.
_busy = [(i, u) for i, u in enumerate(per_gpu_used_mib) if u > FOREIGN_MIB]
if _busy:
    _detail = ", ".join(f"GPU {i}: {u} MiB" for i, u in _busy)
    # Name the culprits rather than guessing at them.
    _apps = _probe_compute_apps()
    if _apps:
        _who = "  Held by:\n" + "\n".join(
            f"    PID {pid:>7}  {mem:>6} MiB  {proc}" for pid, mem, proc in _apps
        )
    else:
        _who = ("  nvidia-smi lists no compute processes for this memory — it may belong to\n"
                "  a container you cannot see from here. Restarting the app (Instance Options\n"
                "  -> Apply & Restart) is then the reliable way to get clean GPUs.")
    raise SystemExit(
        "\n"
        "==================================================================\n"
        " GPU MEMORY IS ALREADY IN USE BY ANOTHER PROCESS — cannot start.\n"
        "==================================================================\n"
        f"  Occupied now: {_detail}\n"
        f"{_who}\n"
        "  The usual cause is M2 (Cosmos Reason Captioning) or M8 (Cosmos Reason\n"
        "  LoRA SFT): both load a 7B model with device_map=\"auto\", which spreads\n"
        "  it across EVERY GPU and holds it for the life of that kernel. Each of\n"
        "  those notebooks now ends with a \"Release the GPUs\" cell — run it there\n"
        "  and this guard passes without shutting anything down.\n\n"
        f"  Cosmos Predict needs nearly the whole GPU (~{MIN_PER_GPU_GB:.0f} GB per\n"
        "  card), so this run would crash with a CUDA out-of-memory error deep\n"
        "  inside the model — NOT\n"
        "  something you can fix by editing this notebook.\n\n"
        "  TO FIX — free the GPUs, then Run All Cells again:\n"
        "    1. CHEAPEST: if the process above is M2's or M8's kernel, open that\n"
        "       notebook and run its last \"Release the GPUs\" cell. Nothing else is\n"
        "       lost, and you can come straight back here.\n"
        "    2. OTHERWISE: JupyterLab top menu -> Kernel -> Shut Down All Kernels\n"
        "       (or the left sidebar 'Running Terminals and Kernels' tab ->\n"
        "        'Shut Down All' under KERNELS).\n"
        "    3. Confirm the cards are free — in a terminal run:\n"
        "         nvidia-smi --query-gpu=memory.used --format=csv,noheader\n"
        "       every GPU should read a few MiB (near 0).\n"
        "    4. Then Run All Cells again (Run menu) — NOT just this cell. A kernel shutdown clears the setup cell above (where MAX_FRAMES etc. are defined), so re-running only this cell fails; Run All re-creates what the guard needs.\n"
        "=================================================================="
    )

# Predict's --resolution takes "H,W"; None => the model's native trained res.
# Predict's frame cap is --num-output-frames (default 77).
if MIN_PER_GPU_GB >= 70:          # H100 80GB (p5)
    RUN_MODE = "single"
    RUN_RESOLUTION = None         # native
    RUN_NUM_FRAMES = None         # default 77
    USE_GUARDRAILS = True
    _why = "H100-class GPU: native-resolution Video2World on a single GPU, guardrails on."
elif MIN_PER_GPU_GB >= 38:        # A100 40GB (p4d)
    RUN_MODE = "shard"
    RUN_RESOLUTION = None         # native
    RUN_NUM_FRAMES = None
    USE_GUARDRAILS = True
    _why = f"A100-class GPUs: native resolution, sharded across {GPU_COUNT} GPUs, guardrails on."
else:                             # L4/A10G 24GB (g5/g6)
    RUN_MODE = "shard"
    RUN_RESOLUTION = "480,832"    # H,W — smaller activations to fit 24GB
    RUN_NUM_FRAMES = 45           # fewer output frames = less activation memory
    USE_GUARDRAILS = False
    _why = (f"24GB-class GPUs: 480x832, {45} output frames, sharded across "
            f"{GPU_COUNT} GPUs, guardrails OFF (they don't fit alongside the "
            f"pipeline here). For native-resolution / longer output, relaunch on "
            f"ml.p4d.24xlarge (A100 40GB) or ml.p5.48xlarge (H100 80GB).")

if RUN_MODE == "shard" and GPU_COUNT < 2:
    raise RuntimeError(
        f"This GPU has only {MIN_PER_GPU_GB:.0f} GB and there is just {GPU_COUNT} GPU, "
        "so Cosmos Predict 2.5-2B cannot be sharded to fit. Pick a multi-GPU "
        "instance (g5.12xlarge / g6.12xlarge / p4d.24xlarge) in Instance Options."
    )

print(f"\nRun strategy: mode={RUN_MODE}, "
      f"resolution={RUN_RESOLUTION or 'native'}, "
      f"frames={RUN_NUM_FRAMES or 'default(77)'}, "
      f"guardrails={'on' if USE_GUARDRAILS else 'off'}")
print(f"  -> {_why}")
print("GPU check PASSED.")

In [ ]:
# ============================================================
# Install the Cosmos Predict 2.5 environment (idempotent)
# ============================================================
# Runs scripts/setup_cosmos_env.sh with the "predict" argument, which:
#   - clones the cosmos-predict2.5 repo + uv-syncs its OWN venv,
#   - applies the SMD-image fixes (opencv-headless, CUDA .so symlinks, ldconfig),
#   - RESTORES the admin's pre-cached Cosmos checkpoints from S3 into HF_HOME,
#     VERIFIES the restored tree holds what this run needs, and writes
#     cosmos_predict_env.sh with HF_HUB_OFFLINE=1 (so inference needs NO HF token),
#   - and REFUSES TO START (exit 2) if there is neither a usable cache nor a token,
#     instead of spending 15-20 min to reach a failure it already knows is coming.
# Re-running after an app restart is safe. First run: ~15-20 min.

# Locate the setup script (repo layout: notebooks/ sits next to scripts/). Guard
# each probe with try/except: some candidate dirs (e.g. /root) raise
# PermissionError from exists() for a non-root user.
def _find_setup_script():
    for base in [Path.cwd(), Path.cwd().parent, Path.home()]:
        cand = base / "scripts" / "setup_cosmos_env.sh"
        try:
            if cand.exists():
                return str(cand)
        except OSError:
            continue
    return None

setup_script = _find_setup_script()
if setup_script is None:
    # Fall back to a copy staged in S3 (notebook-templates ships scripts/ too).
    local = f"{WORK}/setup_cosmos_env.sh"
    subprocess.run(
        ["aws", "s3", "cp",
         f"s3://{SHARED_BUCKET}/notebook-templates/scripts/setup_cosmos_env.sh", local],
        check=True,
    )
    setup_script = local

print(f"Running setup script (predict): {setup_script}")
print("(first run: 15-20 min for uv sync + HF cache restore; re-runs are fast)\n")

env = {**os.environ, "SHARED_BUCKET": SHARED_BUCKET}
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN
proc = subprocess.run(["bash", setup_script, "predict"], env=env, text=True)
if proc.returncode == 2:
    # Exit 2 is the setup script's precondition gate (no usable offline cache AND no
    # token), distinct from exit 1 ("a stack failed to build"). It already printed the
    # reason, the S3 URI it looked for, and both remedies — and it stopped BEFORE
    # spending 15-20 minutes, so nothing is wasted.
    raise RuntimeError(
        "setup_cosmos_env.sh stopped before doing any work: this region has no usable "
        "offline HuggingFace cache and no HF_TOKEN is set. Scroll up for the exact S3 "
        "URI and the two ways to fix it."
    )
if proc.returncode != 0:
    raise RuntimeError(
        "setup_cosmos_env.sh predict failed — scroll up. Common causes: no S3 HF "
        "cache AND no HF_TOKEN, or not enough NVMe space."
    )
# ------------------------------------------------------------
# Cache verdict. Uses a GLOB over $HF_HOME/hub — NOT a substring search of the
# generated env file, whose unquoted heredoc means "export HF_HUB_OFFLINE=1" is
# literally always present there and any read_text() check is always True.
# ------------------------------------------------------------
COSMOS_OFFLINE_OK = bool(glob.glob(f"{NVME}/hf/hub/models--nvidia--Cosmos-*"))
if COSMOS_OFFLINE_OK:
    print("\nCosmos Predict 2.5 environment ready — offline HF cache present, no token needed.")
elif HF_TOKEN:
    print("\nCosmos Predict 2.5 environment ready — NO offline cache; HuggingFace will download")
    print("online with your HF_TOKEN (needs the gated licenses accepted on that account).")
else:
    print("\n*** WARNING: no offline cache and no HF_TOKEN ***")
    print(f"    Expected: s3://{SHARED_BUCKET}/hf-cache/hub/ (region {AWS_REGION})")
    print("    Inference will die within seconds on a gated-repo refusal.")
    print("    Ask your admin to seed hf-cache/ in this region, or set HF_TOKEN in cell 1.")

In [ ]:
# ============================================================
# Obtain the input driving clip (reuse M5's, or rebuild from M1)
# ============================================================
# Video2World needs a source clip. M5 already assembled one from nuScenes
# CAM_FRONT and uploaded it to m5/source/. Reuse it; if M5 wasn't run, rebuild
# from M1's manifest (same logic as M5 cell 4).
#
# This cell runs in the JupyterLab KERNEL (SMD python 3.12), which does NOT ship
# cv2. Install the headless build with --no-deps so it can't upgrade numpy and
# break the SMD image's numpy<2.4 packages.
try:
    import cv2
except ModuleNotFoundError:
    print("Installing opencv-python-headless (--no-deps) into the notebook kernel...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
         "opencv-python-headless"],
        check=True,
    )
    import cv2
print(f"cv2 {cv2.__version__} ready in kernel")

s3 = boto3.client("s3")

# 1) Try to reuse the clip M5 built.
reused = False
try:
    s3.download_file(S3_BUCKET, M5_SOURCE_KEY, INPUT_MP4)
    reused = True
    print(f"Reused M5 clip: s3://{S3_BUCKET}/{M5_SOURCE_KEY}")
except Exception as e:  # noqa: BLE001 — fall back to rebuilding from M1
    print(f"No M5 clip ({e}); rebuilding from M1 CAM_FRONT frames...")

# 2) Fallback: rebuild from M1's manifest.
if not reused:
    m1_manifest_key = f"{M1_PREFIX}manifest.json"
    try:
        body = s3.get_object(Bucket=S3_BUCKET, Key=m1_manifest_key)["Body"].read()
        cam_front_files = json.loads(body).get("cam_front_files", [])
    except s3.exceptions.NoSuchKey:
        raise RuntimeError(f"No M5 clip and no M1 manifest — run M1 first ({m1_manifest_key})")
    assert cam_front_files, "M1 manifest has no cam_front_files — re-run M1"

    frames_local = []
    for rel in cam_front_files[:MAX_FRAMES]:
        dest = os.path.join(FRAMES_DIR, os.path.basename(rel))
        if not os.path.exists(dest):
            s3.download_file(SHARED_BUCKET, f"{NUSCENES_PREFIX}{rel}", dest)
        frames_local.append(dest)

    vw = cv2.VideoWriter(
        INPUT_MP4, cv2.VideoWriter_fourcc(*"mp4v"), VIDEO_FPS, (VIDEO_W, VIDEO_H)
    )
    n = 0
    for f in frames_local:
        img = cv2.imread(f)
        if img is None:
            continue
        vw.write(cv2.resize(img, (VIDEO_W, VIDEO_H)))
        n += 1
    vw.release()
    print(f"Rebuilt input clip from {n} M1 CAM_FRONT frames")

assert os.path.exists(INPUT_MP4) and os.path.getsize(INPUT_MP4) > 0, "No input clip"
print(f"Input clip: {INPUT_MP4} ({os.path.getsize(INPUT_MP4)/1e6:.2f} MB)")

In [ ]:
# ============================================================
# Build Cosmos Predict inference spec(s) — Video2World
# ============================================================
# examples/inference.py takes a JSON spec per sample. For Video2World the spec
# is {inference_type, name, prompt, input_path}; passing a video makes the model
# predict future frames from it. One spec per scenario.
spec_paths = []
for scen in SCENARIOS:
    spec = {
        "inference_type": "video2world",
        "name": f"nuscenes_{scen}",
        "prompt": SCENARIO_PROMPTS[scen],
        "input_path": INPUT_MP4,
    }
    spec_path = f"{SPECS_DIR}/nuscenes_{scen}_spec.json"
    with open(spec_path, "w") as f:
        json.dump(spec, f, indent=2)
    spec_paths.append(spec_path)
    print(f"  {scen}: {spec_path}")

print(f"\nBuilt {len(spec_paths)} spec(s) for scenarios: {SCENARIOS}")

In [ ]:
# ============================================================
# Run Cosmos Predict Video2World inference (instance-adaptive)
# ============================================================
# Uses the strategy chosen in the pre-flight cell (single vs torchrun shard,
# resolution, output-frame count, guardrails). Sources cosmos_predict_env.sh
# (NOT M5's cosmos_env.sh — different package/venv).
generation_start = time.time()
results_manifest = []

def _free_gpus():
    """Reclaim GPU memory before launching inference.

    A failed/interrupted torchrun leaves some ranks alive, each holding ~16 GB
    of GPU memory. A later run then starts on a GPU that already looks full and
    OOMs — even though the code is fine. So before each launch we kill any stale
    Cosmos inference workers (matched by the repo path, so the JupyterLab kernel
    and other users' processes are never touched) and clear our own CUDA cache.
    """
    import subprocess as _sp
    for pat in ("cosmos-predict2.5/examples/inference.py",
                "cosmos-transfer2.5/examples/inference.py"):
        _sp.run(["pkill", "-9", "-f", pat], capture_output=True)
    time.sleep(3)
    try:
        import torch as _t
        if _t.cuda.is_available():
            _t.cuda.empty_cache()
    except Exception:
        pass

def _extra_flags():
    flags = ["--inference-type=video2world"]
    if RUN_RESOLUTION:
        flags += ["--resolution", RUN_RESOLUTION]
    if RUN_NUM_FRAMES:
        flags += ["--num-output-frames", str(RUN_NUM_FRAMES)]
    if not USE_GUARDRAILS:
        flags += ["--disable-guardrails"]
    return " ".join(flags)

for scen, spec_path in zip(SCENARIOS, spec_paths):
    _free_gpus()   # reclaim any GPU memory held by a previous/failed run
    print(f"\n=== Generating '{scen}'  (mode={RUN_MODE}, "
          f"{RUN_RESOLUTION or 'native'}, guardrails={'on' if USE_GUARDRAILS else 'off'}) ===")
    t0 = time.time()

    if RUN_MODE == "single":
        launcher = "python"
    else:
        launcher = f"torchrun --nproc_per_node={GPU_COUNT} --master_port=12357"

    inner = (
        f'export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True && '
        f'source "{COSMOS_PREDICT_ENV}" && cd "{COSMOS_PREDICT_REPO}" && '
        f'{launcher} examples/inference.py -i "{spec_path}" -o "{OUTPUT_DIR}" '
        f'{_extra_flags()}'
    )
    # stderr merged into stdout so cause and effect survive together.
    proc = subprocess.run(["bash", "-lc", inner], text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = "\n".join((proc.stdout or "").splitlines()[-15:])
    print(tail)
    if proc.returncode != 0:
        # --- Failure report -------------------------------------------------
        # See M5 for the full reasoning. In short: the child's real error (e.g. a
        # HuggingFace gated-repo refusal, which cosmos emits via an uncaptured
        # subprocess.check_call) lands on stderr, and slicing stderr to 25 lines
        # could not reach past torchrun's own ~35-line epilogue. Merge the
        # streams, widen the window, save the whole thing, and diagnose from
        # facts rather than asserting CUDA OOM.
        failed_after = time.time() - t0
        combined = proc.stdout or ""
        log_path = os.path.join(OUTPUT_DIR, f"FAILED_{scen}.log")
        try:
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            with open(log_path, "w") as _f:
                _f.write(combined)
            saved = f"Full output: {log_path}"
        except OSError as _e:
            saved = f"(could not write {log_path}: {_e})"
        window = "\n".join(combined.splitlines()[-200:])

        # These are the strings the hf CLI actually prints. It never emits "401",
        # "gated" or "GatedRepo" -- matching those would miss the real failure.
        low = combined.lower()
        gated = any(m in low for m in ("requires approval", "access denied"))
        offline_miss = any(m in low for m in ("local entry not found", "offline mode is enabled"))
        oom = "out of memory" in low or "cuda oom" in low

        if gated:
            why = ("CAUSE: a gated HuggingFace repo was refused. Either the offline "
                   "cache is missing/incomplete in this region, or HF_TOKEN's account "
                   "has not accepted the licenses. THIS IS NOT A GPU PROBLEM — a "
                   "bigger instance will fail identically.")
        elif offline_miss:
            why = ("CAUSE: HuggingFace is in OFFLINE mode and a checkpoint is not in "
                   "the local cache — the restored cache is PARTIAL, which is worse "
                   "than none. Ask your admin to re-seed hf-cache/ in this region.")
        elif oom:
            why = (f"CAUSE: CUDA out of memory. The 480x832+shard path should fit 24 GB — "
                   f"check that all {GPU_COUNT} GPUs were used. For native resolution, "
                   f"relaunch on p4d.24xlarge / p5.48xlarge.")
        elif failed_after < 120:
            why = (f"It failed after only {failed_after:.0f}s, so it never reached "
                   f"the main compute. That rules OUT GPU memory — a bigger instance "
                   f"will not help. Look for an auth, download or import failure.")
        else:
            why = (f"It ran for {failed_after:.0f}s before failing. Read the output "
                   f"above.")

        raise RuntimeError(
            f"Cosmos Predict failed for '{scen}' after {failed_after:.0f}s "
            f"(exit {proc.returncode}).\n"
            f"HF_TOKEN set: {bool(HF_TOKEN)} | offline cache present: "
            f"{bool(glob.glob(f'{NVME}/hf/hub/models--nvidia--Cosmos-*'))}\n"
            f"{saved}\n\n{why}\n\n--- last 200 lines ---\n{window}"
        )

    elapsed = time.time() - t0
    matches = sorted(glob.glob(os.path.join(OUTPUT_DIR, f"**/nuscenes_{scen}*.mp4"),
                               recursive=True))
    gen_mp4 = matches[0] if matches else None
    assert gen_mp4, f"Expected output mp4 not found for '{scen}' under {OUTPUT_DIR}"

    print(f"  -> {gen_mp4} ({os.path.getsize(gen_mp4)/1e6:.2f} MB) in {elapsed:.0f}s")
    results_manifest.append({
        "scenario": scen,
        "source": "nuscenes_cam_front",
        "generated_video": os.path.relpath(gen_mp4, OUTPUT_DIR),
        "prompt": SCENARIO_PROMPTS[scen],
        "resolution": RUN_RESOLUTION or "native",
        "run_mode": RUN_MODE,
        "guardrails": USE_GUARDRAILS,
        "generation_time_s": round(elapsed, 1),
    })

generation_elapsed = time.time() - generation_start
print(f"\nSynthesis complete: {len(results_manifest)} scenario(s) in {generation_elapsed:.0f}s")

In [ ]:
# ============================================================
# Upload Results to S3
# ============================================================
output_s3_path = f"s3://{S3_BUCKET}/{OUTPUT_PREFIX}"
print(f"Uploading synthetic scenarios to {output_s3_path}...")

result = subprocess.run(
    ["aws", "s3", "sync", OUTPUT_DIR, output_s3_path, "--quiet"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f"Upload failed: {result.stderr}")

# Keep the source clip alongside, so the synthesis is reproducible.
subprocess.run(
    ["aws", "s3", "cp", INPUT_MP4, f"{output_s3_path}source/nuscenes_cam_front.mp4",
     "--quiet"],
    capture_output=True, text=True,
)

manifest = {
    "module": "M6_Cosmos_Predict_Synthesis",
    "profile": PROFILE,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model": "Cosmos-Predict2.5-2B (base, video2world)",
    "source": "nuScenes CAM_FRONT (from M5 / M1)",
    "scenarios": SCENARIOS,
    "scenarios_generated": len(results_manifest),
    "results": results_manifest,
}
manifest_path = os.path.join(OUTPUT_DIR, "manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
subprocess.run(
    ["aws", "s3", "cp", manifest_path, f"{output_s3_path}manifest.json", "--quiet"],
    capture_output=True, text=True,
)
print(f"Uploaded {len(results_manifest)} scenario(s) + source + manifest")

In [ ]:
# ============================================================
# Cost Analysis
# ============================================================
# Reads the ACTUAL instance type from SageMaker resource metadata so the estimate
# matches whatever GPU box you launched (g5/g6/p4d/p5).
# Rates are GENERATED PER REGION (scripts/refresh_instance_rates.py, from the AWS Price List
# API) and staged into your workspace, so this estimate uses YOUR region's prices. This was a
# hardcoded us-west-2 table, which understated cost by ~23% in ap-northeast-2 (g6.24xlarge is
# $8.344/hr in Oregon and $10.26/hr in Seoul) — the wrong direction for a cost readout.
import sys
for _rb in (Path.cwd(), Path.cwd().parent, Path.home()):
    if (_rb / "scripts" / "av30_instance_rates.py").exists():
        sys.path.insert(0, str(_rb / "scripts"))
        break
INSTANCE_RATES = {}
try:
    import boto3
    from av30_instance_rates import RATES_BY_REGION
    _region = boto3.Session().region_name
    INSTANCE_RATES = RATES_BY_REGION.get(_region, {})
    if not INSTANCE_RATES:
        print(f"[cost] no generated rate table for {_region}; cost will be omitted. "
              f"Admin: ./scripts/refresh_instance_rates.py --region {_region} --merge")
except Exception as _e:
    print(f"[cost] rate table unavailable ({_e}); cost will be omitted.")
USD_TO_KRW = 1370

inst = "unknown"
try:
    md = json.loads(Path("/opt/ml/metadata/resource-metadata.json").read_text())
    inst = md.get("InstanceType", "unknown")
except Exception:
    pass
rate = INSTANCE_RATES.get(inst)

gen_seconds = generation_elapsed
hours = gen_seconds / 3600

print("=" * 50)
print("COST ANALYSIS — M6 Cosmos Predict Synthesis")
print("=" * 50)
print(f"Instance:        {inst}")
print(f"Run strategy:    {RUN_MODE}, {RUN_RESOLUTION or 'native'} res, "
      f"guardrails={'on' if USE_GUARDRAILS else 'off'}")
print(f"Generation time: {gen_seconds:.0f}s ({hours:.3f} hr)")
if rate is not None:
    cost_usd = hours * rate
    print(f"Rate:            ${rate}/hr")
    print(f"Estimated cost:  ${cost_usd:.2f} USD / {cost_usd * USD_TO_KRW:,.0f} KRW")
    print(f"Scenarios produced: {len(results_manifest)}")
    if results_manifest:
        print(f"Cost per scenario:  ${cost_usd / len(results_manifest):.3f} USD")
else:
    print(f"Rate:            (unknown instance '{inst}' — not in rate table)")
    print(f"Scenarios produced: {len(results_manifest)}")
print("=" * 50)
print("Note: the one-time env install + checkpoint download (setup cell) adds")
print("~15-25 min of instance time on the FIRST run of a fresh app.")
print("Tip: 24GB GPUs (g5/g6) run 480x832; p4d/p5 run native resolution.")

In [ ]:
# ============================================================
# Output Validation + Next Module
# ============================================================
gen_files = [os.path.join(OUTPUT_DIR, r["generated_video"]) for r in results_manifest]
print("Output validation:")
print(f"  Generated scenarios: {len(gen_files)} (expected {len(SCENARIOS)})")

ok = True
for f in gen_files:
    exists = os.path.exists(f)
    size_mb = os.path.getsize(f) / 1e6 if exists else 0
    flag = "OK" if (exists and size_mb > 0.1) else "MISSING/TOO SMALL"
    print(f"    {os.path.basename(f)}: {size_mb:.2f} MB [{flag}]")
    ok = ok and exists and size_mb > 0.1

print(f"  Status: {'PASS' if ok and gen_files else 'FAIL'}")
print(f"\nOutput location: {output_s3_path}")

# Preview the first generated clip inline (optional).
try:
    from IPython.display import Video, display
    if gen_files and os.path.exists(gen_files[0]):
        display(Video(gen_files[0], embed=True, width=640))
except Exception as e:
    print(f"(inline preview skipped: {e})")

print("\n" + "=" * 50)
print("NEXT MODULE")
print("=" * 50)
print("M7: Nerfstudio 3D Reconstruction  (next in blog-stage order)")
print("  Stage 6 — rebuilds a 3D scene from the nuScenes CAM_FRONT images.")
print("  Independent branch: reads the shared dataset, not this module's output.")
print("  Instance: ml.g5.xlarge")
print("")
print("Later in the policy branch: M9 Alpamayo VLA inference (ml.g6.24xlarge),")
print("then M10 AlpaSim closed-loop evaluation.")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    mark_complete("m06-cosmos-predict")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")